## Visão Geral do Okta

Okta é um serviço de gerenciamento de identidade e acesso baseado em nuvem que fornece soluções de identidade seguras para empresas, possibilitando autenticação e autorização contínuas em aplicações e serviços.

Recursos Principais:
* **Single Sign-On (SSO)** - Usuários se autenticam uma vez para acessar múltiplas aplicações
* **Multi-Factor Authentication (MFA)** - Segurança aprimorada através de métodos adicionais de verificação
* **Adaptive Authentication** - Políticas de autenticação baseadas em risco com base no comportamento e contexto do usuário
* **Universal Directory** - Gerenciamento centralizado de usuários e sincronização de perfis
* **API Access Management** - Suporte OAuth 2.0 e OpenID Connect para segurança de API

## Objetivo de Aprendizado
Okta pode ser usado como provedor de identidade no AgentCore Identity e usado para autenticar usuários e fazê-los autorizar o agente a acessar recursos protegidos em seu nome. Neste notebook exploraremos o uso do Okta para autenticação de entrada - Autenticar usuários antes que possam invocar um agente.

## Fluxo Authorization Code
O fluxo authorization code do OAuth 2.0 é a abordagem recomendada para aplicações web autenticarem usuários com segurança e obterem tokens de acesso. Este fluxo envolve:
1. Redirecionar usuários para o Okta para autenticação
2. Receber um código de autorização após login bem-sucedido
3. Trocar o código por tokens de acesso e refresh
4. Usar tokens para acessar recursos protegidos

Este padrão de integração permite que o AgentCore aproveite as robustas capacidades de gerenciamento de identidade do Okta enquanto mantém autenticação segura e baseada em padrões para suas aplicações.

## Arquitetura do Tutorial

```
┌──────────┐  1. Credenciais  ┌──────────┐  2. Token JWT  ┌──────────┐
│  Client  │ ───────────────► │   Okta   │ ─────────────► │  Client  │
└──────────┘                  └──────────┘                └──────────┘
                                                                 │
                                                                 │ 3. Bearer JWT
                                                                 ▼
┌──────────┐                  ┌──────────┐                ┌──────────┐
│  Client  │ ◄─────────────── │ Bedrock  │ ─────────────► │  Agent   │
└──────────┘  6. Resposta     │AgentCore │  4. Invocar    └──────────┘
                              └──────────┘                      │
                                    ▲                           │
                                    └─────── 5. Resposta ───────┘
```

<figure>
    <img src="images/16.png">
</figure>

## Detalhes do Tutorial

| Informação         | Detalhes                                                                         |
|:-------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial   | Conversacional                                                                   |
| Tipo de agente     | Único                                                                            |
| Framework agêntico | Strands Agents                                                                   |
| Modelo LLM         | Anthropic Claude Sonnet 3.5                                                     |
| Componentes        | Hospedar agente no AgentCore Runtime. Usando Strands Agent e Amazon Bedrock Model |
| Vertical           | Cross-vertical                                                                   |
| Complexidade       | Fácil                                                                            |
| Inbound Auth       | Okta                                                                             |
| SDK usado          | Amazon BedrockAgentCore Python SDK e boto3                                      |

### Recursos Principais

* Hospedar Agentes no Amazon Bedrock AgentCore Runtime com Inbound Auth usando Okta
* Usar modelos Amazon Bedrock
* Usar Strands Agents

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Direitos IAM para criar novas roles, políticas e usuários IAM
* Direitos IAM para criar um novo AgentCore Agent
* Uma conta Okta
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker rodando

## Objetivo de Aprendizado 1: Configurar Okta para uso com AgentCore

## Passo 1: Configurando o IDP do Okta

Antes de configurarmos nosso AgentCore Runtime com autenticação Okta, precisamos configurar o Okta como nosso Provedor de Identidade. Esta seção guiará você através da criação de um tenant Okta, configuração de uma aplicação e configuração dos usuários e reivindicações necessárias.

### 1.1 Criar Conta Okta Developer

Se você ainda não tem uma conta Okta, navegue até https://developer.okta.com/signup/ e selecione "Sign up for Integrator Free Plan" para se cadastrar.

### 1.2 Adicionar um Usuário de Teste

1. Faça login em sua conta Okta.
2. Selecione **Directory**, depois **People** e clique em **Add person**.

   <figure>
       <img src="images/9.png">
   </figure>

3. Preencha o formulário:
   - Para **Activation**, selecione **Activate now**.
   - Marque **I will set password** e defina uma senha para o usuário.
   - Desmarque **User must change password on first login**.
   - Clique em **Save**.

   <figure>
       <img src="images/10.png">
   </figure>

### 1.3 Criar Integração de Aplicação

4. Selecione **Applications**, depois clique em **Create App Integration**.

   <figure>
       <img src="images/1.png">
   </figure>

5. Para o método de sign-in, selecione **OIDC - OpenID Connect**, depois selecione **Web Application** para o tipo de aplicação.

   <figure>
       <img src="images/2_enhanced.png">
   </figure>

6. Configure a aplicação:
   - Para App integration name insira **AgentCore Inbound Auth**
   - Selecione **Authorization Code** para o tipo de grant.
   - Use "https://bedrock-agentcore.us-west-2.amazonaws.com/identities/oauth2/callback" ou "https://bedrock-agentcore.us-east-1.amazonaws.com/identities/oauth2/callback" como URL de redirecionamento dependendo de qual região seu agente estará rodando.

   <figure>
       <img src="images/3_enhanced.png">
   </figure>

   - Em assignments, selecione **Allow everyone in your organization to access**, depois deixe **Enable immediate access** marcado. Clique em **Save**.

   <figure>
       <img src="images/5_enhanced.png">
   </figure>

   - Copie o **Client ID** e **Secret** para uso posterior.

   <figure>
       <img src="images/6_enhanced.png">
   </figure>

### 1.4 Configurar Servidor de Autorização

7. No menu do lado esquerdo, selecione **Security**, depois **API**, e clique no nome do seu servidor de autorização.

   <figure>
       <img src="images/7_enhanced.png">
   </figure>

   - Copie o **Audience** e salve-o para uso posterior.
     > **Nota**: O **Audience** padrão foi alterado neste exemplo. É recomendado adicionar um novo servidor de autorização se você planeja alterar o audience para que outras aplicações não sejam afetadas.
   
   - Clique em **Scopes** e adicione um novo scope chamado **agentcore**.

   <figure>
       <img src="images/agencore.png">
   </figure>

   - Clique em **Claims** e adicione as seguintes reivindicações **client_id** e **scope**.

   <figure>
       <img src="images/8.png">
   </figure>

### 1.5 Coletar Valores de Configuração

Após completar a configuração do Okta, você deve ter os seguintes valores:

- **OKTA_CLIENT_ID**: Client ID da aplicação da aba General
- **OKTA_CLIENT_SECRET**: Client Secret da aplicação da aba General  
- **OKTA_AUDIENCE**: Audience (ex.: `testagentcore`)
- **OKTA_TOKEN_URL**: Seu domínio Okta + `/oauth2/default/v1/token`
- **OKTA_DISCOVERY_URL**: Seu domínio Okta + `/oauth2/default/.well-known/openid-configuration`

Mantenha esses valores à mão pois você precisará deles nos próximos passos.

Nota:
1. Okta não é um serviço AWS. Consulte a documentação do Okta para custos relacionados ao Okta.
2. Capturas de tela usadas nos seguintes passos podem mudar. Encorajamos você a consultar a documentação do Okta para orientações mais recentes sobre configuração de aplicação Okta.

## Objetivo de Aprendizado 2 - Configurar um agente simples com Okta para autenticação de entrada

#### Pré-requisitos
1. Instalar pacotes necessários
2. Importar pacotes
3. Obter account ID para usar ao longo do notebook
4. Definir região AWS como "us-west-2". Você pode usar qualquer região que suporte Bedrock AgentCore. Consulte https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html

## Passo 2: Configuração do Ambiente

Primeiro, vamos configurar o ambiente de desenvolvimento e instalar as dependências necessárias.

In [ ]:
# Create and activate virtual environment
!python -m venv .venv
!source .venv/bin/activate

Para este código funcionar, os módulos Strands Agents precisam estar instalados no ambiente Python.

Adicione os módulos Strands Agents, AgentCore SDK e AgentCore starter toolkit ao arquivo de dependências e salve-o como **requirements.txt**:

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit
PyJWT

In [ ]:
# Install required packages from requirements.txt
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Verify that all required packages are installed correctly
try:
    import bedrock_agentcore
    import strands
    print("✅ All packages installed successfully")
    print("✅ bedrock-agentcore: imported")
    print("✅ strands: imported")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure all packages are installed correctly")

## Passo 3: Configurar Variáveis de Ambiente


Definindo variáveis de ambiente para algumas informações chave que precisaremos ao longo deste notebook.

In [ ]:
# Okta Configuration - Replace with your values
import os

os.environ['OKTA_CLIENT_ID'] = 'YOUR_CLIENT_ID_VALUE'
os.environ['OKTA_CLIENT_SECRET'] = 'YOUR_CLIENT_SECRET_VALUE'
os.environ['OKTA_AUDIENCE'] = 'YOUR_OKTA_AUDIENCE'
os.environ['OKTA_TOKEN_URL'] = 'https://your.okta.com/oauth2/default/v1/token'
os.environ['OKTA_DISCOVERY_URL'] = 'https://your.okta.com/oauth2/default/.well-known/openid-configuration'

## Passo 4: Código do Agente
Mantendo o agente simples já que o objetivo de aprendizado chave para este notebook é aprender autenticação de entrada usando Okta

In [ ]:
%%writefile simple_agent.py
import argparse, json
from strands import Agent, tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
agent = Agent()

@app.entrypoint
def invoke(payload):
    """Simple agent function for inbound auth demo"""
    user_message = payload.get("prompt", "Hello! How can I help you today?")
    
    # Get session information if available
    session_id = payload.get("session_id", "no-session")
    
    # Simple response with session awareness
    response = f"Hello! I'm a simple agent with session ID: {session_id}. You asked: {user_message}"
    
    result = agent(response)
    return {"result": result.message}

if __name__ == "__main__":
    app.run()

## Passo 5: Configurar AgentCore Runtime com Autenticação Okta

Configure o AgentCore Runtime com autenticação OAuth do Okta.

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# Get account ID and region
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
region = 'us-west-2'

print(f"Account ID: {account_id}")
print(f"Region: {region}")

# Use environment variables
discovery_url = os.getenv('OKTA_DISCOVERY_URL')
client_id = os.getenv('OKTA_CLIENT_ID')
audience = os.getenv('OKTA_AUDIENCE')

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id[:4]}****{client_id[-4:] if client_id else 'None'}")  # Masked
print(f"Audience: {audience}")

agentcore_runtime = Runtime()

# Try with OAuth configuration
try:
    response = agentcore_runtime.configure(
        entrypoint="simple_agent.py",
        auto_create_execution_role=True,
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name="okta_inbound_auth_agent",
        authorizer_configuration={
            "customJWTAuthorizer": {
                "discoveryUrl": discovery_url,
                "allowedClients": [client_id],
                "allowedAudience": [audience]
            }
        }
    )
    print("✅ OAuth configuration successful")
except Exception as e:
    print(f"❌ OAuth configuration failed: {e}")

response

## Passo 6: Lançar agente no AgentCore Runtime

Agora que configuramos o agente, vamos lançá-lo no AgentCore Runtime.

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

## Passo 7: Verificar o Status do AgentCore Runtime

Monitore o status da implantação até que o agente esteja pronto.

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

## Passo 8: Salvar ARN do Agente para Testes

Extrair e salvar o ARN do Agente para fins de teste.

Extrair e salvar ARN do agente para fins de teste

In [ ]:
# Extract ARN from launch_result
if hasattr(launch_result, 'agent_arn') and launch_result.agent_arn:
    agent_arn = launch_result.agent_arn
    os.environ['AGENT_ARN'] = agent_arn
    print(f"📝 Agent ARN: {agent_arn}")
    print(f"📝 Agent ID: {launch_result.agent_id}")
    print(f"📝 ECR URI: {launch_result.ecr_uri}")
else:
    print("⚠️  Could not extract Agent ARN from launch result")
    print("Launch result:", launch_result)

# Check deployment status
if status == 'READY':
    print("✅ Agent deployed successfully and ready for testing!")
elif status in ['CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    print(f"❌ Agent deployment failed with status: {status}")
else:
    print(f"⚠️  Unexpected status: {status}")

## Passo 9: Criar Cliente de Teste

Crie um cliente de teste para validar o fluxo OAuth do Okta e invocação do agente com suporte a sessão.

In [ ]:
import requests
import json
import time
import urllib.parse
import logging
import uuid

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
client_id = os.getenv('OKTA_CLIENT_ID')
client_secret = os.getenv('OKTA_CLIENT_SECRET')
audience = os.getenv('OKTA_AUDIENCE')
token_url = os.getenv('OKTA_TOKEN_URL')
AGENT_ARN = os.getenv('AGENT_ARN')

print("✅ Configuration loaded successfully")

Definir função para obter token de acesso OAuth do Okta

In [ ]:
def get_oauth_token():
    """Get OAuth token from Okta"""
    data = {
        'grant_type': 'client_credentials',
        'scope': 'agentcore'
    }
    
    logger.info("🔐 Getting OAuth token...")
    
    response = requests.post(
        token_url,
        data=data,
        auth=(client_id, client_secret)
    )
    
    response.raise_for_status()
    token_data = response.json()
    logger.info("✅ OAuth token obtained")
    return token_data['access_token']

Definir função para invocar agente com autenticação e suporte a sessão

In [ ]:
def invoke_agent(access_token, query, session_id=None):
    """Invoke agent with session ID support"""
    escaped_agent_arn = urllib.parse.quote(AGENT_ARN, safe='')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
    
    # Generate session ID if not provided
    if not session_id:
        session_id = f'okta-inbound-session-{int(time.time())}-{uuid.uuid4().hex[:8]}'
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id
    }
    
    payload = {
        'prompt': query,
        'session_id': session_id
    }
    
    logger.info(f"🚀 Invoking agent with query: {query}")
    logger.info(f"📋 Session ID: {session_id}")
    
    response = requests.post(url, headers=headers, json=payload, timeout=300)
    response.raise_for_status()
    
    result = response.json()
    logger.info("✅ Agent response received")
    return result, session_id

### Testar Requisição Não Autenticada (Deve Falhar)

Primeiro, vamos verificar se nosso agente rejeita adequadamente requisições não autenticadas:

In [ ]:
# Test unauthenticated request - this should fail
print("=" * 50)
print("TEST: Unauthenticated Request (Should Fail)")
print("=" * 50)

try:
    escaped_agent_arn = urllib.parse.quote(AGENT_ARN, safe='')
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
    
    headers = {
        'Content-Type': 'application/json'
        # Note: No Authorization header
    }
    
    payload = {'prompt': 'Hello without authentication'}
    
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    print(f"❌ Unexpected success! Status: {response.status_code}")
    print(f"Response: {response.text}")
    
except requests.exceptions.HTTPError as e:
    print(f"✅ Expected authentication failure: {e.response.status_code}")
    print(f"Error message: {e.response.text}")
except Exception as e:
    print(f"✅ Expected authentication error: {e}")

print("\n🔒 This confirms that authentication is required!")

Obter token de acesso OAuth do Okta para requisições autenticadas

In [ ]:
# Get OAuth token
access_token = get_oauth_token()
print("✅ OAuth token obtained successfully")

Testar invocação do agente autenticada com rastreamento de session ID

In [ ]:
# Test 1: Simple query with session ID
print("=" * 50)
print("TEST 1: Simple Query with Session ID")
print("=" * 50)

result1, session_id1 = invoke_agent(access_token, "Hello can you guide me about AWS security best practices for authentication?")
print(f"Session ID used: {session_id1}")
print(json.dumps(result1, indent=2))

Testar continuidade de sessão reutilizando o mesmo session ID

In [ ]:
# Test 2: Continue conversation with same session ID
print("=" * 50)
print("TEST 2: Continue Conversation with Same Session")
print("=" * 50)

result2, session_id2 = invoke_agent(access_token, "What was my previous question?", session_id1)
print(f"Session ID used: {session_id2}")
print(json.dumps(result2, indent=2))

Testar validação de scope para acesso não autorizado com nome de scope errado

Scopes definem quais permissões uma aplicação tem - eles fornecem controle de acesso granular para autorização OAuth2

In [ ]:
import jwt

def check_token_scopes(access_token, required_scope='agentcore'):
    """Check if token has required scope"""
    try:
        # Decode token without verification for demo (in production, verify signature)
        decoded = jwt.decode(access_token, options={"verify_signature": False})
        token_scopes = decoded.get('scp', [])
        
        # Check if token has required scope
        has_access = required_scope in token_scopes
        
        return {
            'has_access': has_access,
            'token_scopes': token_scopes,
            'required_scope': required_scope
        }
    except Exception as e:
        return {'error': str(e), 'has_access': False}

# Test 3: Scope validation - Negative scenario
print("=" * 50)
print("TEST 3: Scope Validation - Negative Scenario")
print("=" * 50)

# Check for wrong scope name - 'admin' scope would grant administrative privileges
wrong_scope_check = check_token_scopes(access_token, 'admin')
print(f"Wrong scope check result: {json.dumps(wrong_scope_check, indent=2)}")

if wrong_scope_check.get('has_access'):
    print("✅ Token has required scope")
else:
    print("❌ Access denied: Token does not have required scope")
    print(f"Token has scopes: {wrong_scope_check['token_scopes']}")
    print(f"Required scope: {wrong_scope_check['required_scope']}")
    print("Agent invocation would be blocked at application level")

Testar validação de scope para acesso autorizado com nome de scope correto

Scopes habilitam acesso seguro a API limitando quais ações uma aplicação autenticada pode executar

In [ ]:
# Test 4: Scope validation - Positive scenario
print("=" * 50)
print("TEST 4: Scope Validation - Positive Scenario")
print("=" * 50)

# Check for correct scope - 'agentcore' scope grants access to invoke AgentCore agents
scope_check = check_token_scopes(access_token, 'agentcore')
print(f"Scope check result: {json.dumps(scope_check, indent=2)}")

if scope_check.get('has_access'):
    print("✅ Token has required scope - proceeding with agent call")
    result4, session_id4 = invoke_agent(access_token, "I have the right scope! Tell me about AWS security.")
    print(f"Agent response: {result4}")
else:
    print("❌ Token lacks required scope - access denied")

## Conclusão e Limpeza
Neste notebook aprendemos como:
- Configurar API e Aplicação do Okta para fornecer fluxo OAuth Authorization Code
- Criar um AgentCore Runtime e Implantar um agente com autenticação de entrada usando Okta
- Obter um token e usá-lo para acessar o Agente protegido
- Demonstrar gerenciamento e continuidade de sessão

#### Recurso(s) criado(s)

In [ ]:
# Display the created agent ID
if hasattr(launch_result, 'agent_id'):
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"Agent ARN: {launch_result.agent_arn}")
else:
    print("Agent information not available")

#### Deletar AgentCore Runtime

Limpar os recursos criados durante este tutorial:

In [ ]:
# Delete the AgentCore Runtime
try:
    if hasattr(launch_result, 'agent_id'):
        agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
        agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
        print(f"✅ Agent {launch_result.agent_id} deleted successfully")
    else:
        print("⚠️  No agent ID found to delete")
except Exception as e:
    print(f"❌ Error deleting agent: {e}")
    print("You may need to delete the agent manually from the AWS console")

## Conclusão

Este notebook demonstrou como:

1. **Configurar IDP Okta** - Configurar tenant Okta, aplicação e servidor de autorização
2. **Criar Agente Simples** - Construir um agente básico com consciência de sessão
3. **Configurar Autenticação OAuth** - Configurar AgentCore Runtime com validação JWT do Okta
4. **Implantar com Autenticação** - Implantar agente com autenticação de entrada
5. **Testar Fluxo de Autenticação** - Verificar fluxo de token OAuth e gerenciamento de sessão

### Aprendizados Chave:

- **Autenticação de Entrada**: Okta protege endpoints de agentes, garantindo que apenas usuários autenticados possam invocar agentes
- **Gerenciamento de Sessão**: Agentes podem acessar informações de sessão para respostas personalizadas
- **Validação de Token JWT**: AgentCore valida automaticamente tokens JWT do Okta
- **Segurança**: Requisições não autenticadas são automaticamente rejeitadas

### Próximos Passos:

- Implementar fluxos de autenticação baseados em usuário
- Adicionar lógica de agente mais sofisticada com contexto de usuário
- Explorar autenticação de saída para acessar APIs externas
- Integrar com AgentCore Gateway para camadas adicionais de segurança